model_12_6_0.xlsx', model_13_9_6.xlsx', model_20_5_0.xlsx', model_22_8_7.xlsx', model_24_6_2.xlsx', model_2_8_0.xlsx', model_30_7_8.xlsx', model_9_9_10.xlsx', model_9_9_2.xlsx', model_9_9_5.xlsx'

In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

def redes(file_names,file_names_1000):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names_1000]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_pred_sum = np.sum(Z_pred_total, axis=1).reshape(-1, 1)
    df = pd.read_excel("1000_model/1000_model_12_6_0.xlsx")
    Z = df['Z'].values.reshape(-1, 1)
    mse_sup = np.mean((Z - Z_pred_sum/10) ** 2)

    r2_sup_1000 = r2_score(Z, Z_pred_sum/10)
    var = [np.mean((z_pred - Z_pred_sum/10) ** 2) for z_pred in z_preds]
    var_1000 = np.mean(np.array(var))
    bias_1000 = np.mean((Z_pred_sum/10 - Z))

    M = 10  # número de redes
    cov_sum = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                f_i = z_preds[i]
                f_j = z_preds[j]
                f_i_mean = np.mean(f_i)
                f_j_mean = np.mean(f_j)
                cov_ij = np.mean((f_i - f_i_mean) * (f_j - f_j_mean))
                cov_sum += cov_ij

    covar_1000 = cov_sum / (M * (M - 1))

    df_list1 = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds1 = [df['Z_pred'].values.reshape(-1, 1) for df in df_list1]
    Z_pred_total1 = np.hstack(z_preds1)
    Z_pred_sum1 = np.sum(Z_pred_total1, axis=1).reshape(-1, 1)
    df1 = pd.read_excel("25_model/25_model_12_6_0.xlsx")
    Z1 = df1['Z'].values.reshape(-1, 1)

    mse_sup1 = np.mean((Z1 - Z_pred_sum1/10) ** 2)

    r2_sup_25 = r2_score(Z1, Z_pred_sum1/10)
    var = [np.mean((z_pred - Z_pred_sum1/10) ** 2) for z_pred in z_preds1]
    var_25 = np.mean(np.array(var))
    bias_25 = np.mean((Z_pred_sum1/10 - Z1))

    M = 10  # número de redes
    cov_sum1 = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                f_i = z_preds1[i]
                f_j = z_preds1[j]
                f_i_mean = np.mean(f_i)
                f_j_mean = np.mean(f_j)
                cov_ij = np.mean((f_i - f_i_mean) * (f_j - f_j_mean))
                cov_sum1 += cov_ij

    covar_25 = cov_sum1 / (M * (M - 1))
    
    tabela = pd.DataFrame({
    "Métrica": ["R²", "MSE", "Variância", "Bias", "Covariância"],
    "1024 dados": [r2_sup_1000, mse_sup, var_1000, bias_1000, covar_1000],
    "25 dados": [r2_sup_25, mse_sup1, var_25, bias_25, covar_25]
})

    print(tabela)

    return mse_sup


file_names_1000 =  ["1000_model/1000_model_12_6_0.xlsx", "1000_model/1000_model_13_9_6.xlsx",
              "1000_model/1000_model_20_5_0.xlsx", "1000_model/1000_model_22_8_7.xlsx", 
              "1000_model/1000_model_24_6_2.xlsx", "1000_model/1000_model_2_8_0.xlsx", 
              "1000_model/1000_model_30_7_8.xlsx","1000_model/1000_model_9_9_0.xlsx",
              "1000_model/1000_model_9_9_2.xlsx", "1000_model/1000_model_9_9_5.xlsx"]

file_names = [
    "25_model/25_model_12_6_0.xlsx",
    "25_model/25_model_13_9_6.xlsx",
    "25_model/25_model_20_5_0.xlsx",
    "25_model/25_model_22_8_7.xlsx",
    "25_model/25_model_24_6_2.xlsx",
    "25_model/25_model_2_8_0.xlsx",
    "25_model/25_model_30_7_8.xlsx",
    "25_model/25_model_9_9_0.xlsx",
    "25_model/25_model_9_9_2.xlsx",
    "25_model/25_model_9_9_5.xlsx"
]
result= redes(file_names, file_names_1000)




       Métrica  1024 dados  25 dados
0           R²    0.838076  0.994027
1          MSE    0.679561  0.025068
2    Variância    0.129909  0.067797
3         Bias    0.004191  0.005706
4  Covariância    3.858460  3.896305


In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


lambda_reg = 5e-6        
lr = 3e-5                 
n_epocas = 5000000


def redes(file_names):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1,1) for df in df_list])
    dfZ = pd.read_excel("25_model/25_model_12_6_0.xlsx")
    Z = dfZ['Z'].values.reshape(-1, 1)
    N = Z.shape[0]
    return Z, z_preds, N

def pesos(Z, z_preds, N, patience=500, min_delta=10e-3):  
    
    best_erro = np.inf
    best_w = None
    patience_counter = 0
    a = np.random.uniform(0, 1, z_preds.shape[1])

    for epoch in range(1, n_epocas+1):
        w = np.exp(a) / np.sum(np.exp(a)) # softmax
        yhat = np.dot(z_preds, w)
        residuo = Z.flatten() - yhat
        mse = np.mean(residuo**2)
        mse_pond = mse + lambda_reg * np.sum(a**2)
        # gradiente dmse/dw
        gmse = (-2.0 / N) * z_preds.T.dot(residuo)
        # gradiente completo
        s = np.dot(gmse, w)
        grad_a = w * (gmse - s) + 2.0 * lambda_reg * a
        # atualização
        a = a - lr * grad_a

        # ---- EARLY STOPPING ----
        if mse_pond < best_erro - min_delta:
            best_erro = mse_pond
            best_w = w.copy()
            patience_counter = 0  # reset
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"\nEarly stopping ativado na época {epoch} — perda não melhorou por {patience} épocas.")
            break
        # -------------------------

        if epoch % 100 == 0 or epoch == 1:
            r2 = 1 - np.sum((Z - yhat.reshape(-1,1))**2) / np.sum((Z - Z.mean())**2)
        

    return best_erro, best_w

def mse(file_names, file_names_1000, best_w):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names_1000]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    yhat_final = Z_pred_total * best_w      
    Z_pred_sum = np.sum(yhat_final, axis=1).reshape(-1, 1)
    df = pd.read_excel("1000_model/1000_model_12_6_0.xlsx")
    Z = df['Z'].values.reshape(-1, 1)
    mse_sup_1000 = np.mean((Z - Z_pred_sum) ** 2)
    r2_sup_1000 = r2_score(Z, Z_pred_sum)
    var = [np.mean((z_pred - Z_pred_sum/10) ** 2) for z_pred in z_preds]
    var_1000 = np.mean(np.array(var))
    bias_1000 = np.mean((Z_pred_sum/10 - Z))
    M = 10  # número de redes
    cov_sum = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                f_i = z_preds[i]
                f_j = z_preds[j]
                f_i_mean = np.mean(f_i)
                f_j_mean = np.mean(f_j)
                cov_ij = np.mean((f_i - f_i_mean) * (f_j - f_j_mean))
                cov_sum += cov_ij

    covar_25 = cov_sum / (M * (M - 1))

    df_list2 = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds2 = [df['Z_pred'].values.reshape(-1, 1) for df in df_list2]
    Z_pred_total2 = np.hstack(z_preds2)
    yhat_final2 = Z_pred_total2 * best_w      
    Z_pred_sum2 = np.sum(yhat_final2, axis=1).reshape(-1, 1)
    df2 = pd.read_excel("25_model/25_model_12_6_0.xlsx")
    Z2 = df2['Z'].values.reshape(-1, 1)
    mse_25 = np.mean((Z2 - Z_pred_sum2) ** 2)
    r2_sup_25 = r2_score(Z2, Z_pred_sum2)
    var2 = [np.mean((z_pred - Z_pred_sum2/10) ** 2) for z_pred in z_preds2]
    var_25 = np.mean(np.array(var2))
    bias_25 = np.mean((Z_pred_sum2/10 - Z2))

    cov_sum2 = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                f_i = z_preds[i]
                f_j = z_preds[j]
                f_i_mean = np.mean(f_i)
                f_j_mean = np.mean(f_j)
                cov_ij = np.mean((f_i - f_i_mean) * (f_j - f_j_mean))
                cov_sum2 += cov_ij

    covar_1000 = cov_sum2 / (M * (M - 1))

    tabela = pd.DataFrame({
    "Métrica": ["R²", "MSE", "Variância", "Bias", "Covariância"],
    "1024 dados": [r2_sup_1000, mse_sup_1000, var_1000, bias_1000, covar_1000],
    "25 dados": [r2_sup_25, mse_25, var_25, bias_25, covar_25]
})
    print(tabela)

    return mse_sup_1000, r2_sup_1000, mse_25

file_names_1000 =  ["1000_model/1000_model_12_6_0.xlsx", "1000_model/1000_model_13_9_6.xlsx",
              "1000_model/1000_model_20_5_0.xlsx", "1000_model/1000_model_22_8_7.xlsx", 
              "1000_model/1000_model_24_6_2.xlsx", "1000_model/1000_model_2_8_0.xlsx", 
              "1000_model/1000_model_30_7_8.xlsx","1000_model/1000_model_9_9_0.xlsx",
              "1000_model/1000_model_9_9_2.xlsx", "1000_model/1000_model_9_9_5.xlsx"]

file_names = [
    "25_model/25_model_12_6_0.xlsx",
    "25_model/25_model_13_9_6.xlsx",
    "25_model/25_model_20_5_0.xlsx",
    "25_model/25_model_22_8_7.xlsx",
    "25_model/25_model_24_6_2.xlsx",
    "25_model/25_model_2_8_0.xlsx",
    "25_model/25_model_30_7_8.xlsx",
    "25_model/25_model_9_9_0.xlsx",
    "25_model/25_model_9_9_2.xlsx",
    "25_model/25_model_9_9_5.xlsx"
]

Z, z_preds, N= redes(file_names)
best_erro, best_w = pesos(Z, z_preds, N)

print("best erro =", best_erro)
print("best pesos =", best_w)

mse_1000 =mse(file_names, file_names_1000, best_w)




Early stopping ativado na época 501 — perda não melhorou por 500 épocas.
best erro = 0.026886823572133976
best pesos = [0.07400122 0.11792899 0.10885092 0.08895212 0.12916583 0.07154637
 0.0769191  0.12632146 0.09435217 0.11196182]
       Métrica  1024 dados  25 dados
0           R²    0.837891  0.993600
1          MSE    0.680337  0.026860
2    Variância    3.487821  3.451435
3         Bias   -0.465635 -0.465484
4  Covariância    3.858460  3.858460


In [17]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score


def redes(file_names):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list])

    dfZ = pd.read_excel("25_model/25_model_12_6_0.xlsx")
    Z = dfZ['Z'].values.reshape(-1, 1)

    N = Z.shape[0]
    return Z, z_preds, N


def pesos(Z, z_preds, N, lambda_reg, lr, n_epocas, patience=500, min_delta=1e-2):

    best_erro = np.inf
    best_w = None
    patience_counter = 0

    a = np.random.uniform(0, 1, z_preds.shape[1])

    for epoch in range(1, n_epocas + 1):

        w = np.exp(a) / np.sum(np.exp(a))  # softmax
        yhat = z_preds @ w

        residuo = Z.flatten() - yhat
        mse = np.mean(residuo ** 2)
        mse_pond = mse + lambda_reg * np.sum(a ** 2)

        gmse = (-2.0 / N) * z_preds.T @ residuo
        s = gmse @ w
        grad_a = w * (gmse - s) + 2.0 * lambda_reg * a

        a -= lr * grad_a

        if mse_pond < best_erro - min_delta:
            best_erro = mse_pond
            best_w = w.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return best_w


def avalia_ensemble(file_names, Z_path, best_w):
    df_list = [pd.read_excel(f) for f in file_names]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list])

    Z = pd.read_excel(Z_path)['Z'].values.reshape(-1, 1)

    yhat = z_preds @ best_w
    mse = np.mean((Z.flatten() - yhat) ** 2)
    r2 = r2_score(Z, yhat)

    M = z_preds.shape[1]

    var = np.mean([(z_preds[:, i] - yhat) ** 2 for i in range(M)])
    bias = np.mean(yhat - Z.flatten())

    cov_sum = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                cov_sum += np.mean(
                    (z_preds[:, i] - z_preds[:, i].mean()) *
                    (z_preds[:, j] - z_preds[:, j].mean())
                )

    covar = cov_sum / (M * (M - 1))

    return mse, r2, var, bias, covar


file_names_1000 = [
    "1000_model/1000_model_12_6_0.xlsx",
    "1000_model/1000_model_13_9_6.xlsx",
    "1000_model/1000_model_20_5_0.xlsx",
    "1000_model/1000_model_22_8_7.xlsx",
    "1000_model/1000_model_24_6_2.xlsx",
    "1000_model/1000_model_2_8_0.xlsx",
    "1000_model/1000_model_30_7_8.xlsx",
    "1000_model/1000_model_9_9_0.xlsx",
    "1000_model/1000_model_9_9_2.xlsx",
    "1000_model/1000_model_9_9_5.xlsx"
]

file_names_25 = [
    "25_model/25_model_12_6_0.xlsx",
    "25_model/25_model_13_9_6.xlsx",
    "25_model/25_model_20_5_0.xlsx",
    "25_model/25_model_22_8_7.xlsx",
    "25_model/25_model_24_6_2.xlsx",
    "25_model/25_model_2_8_0.xlsx",
    "25_model/25_model_30_7_8.xlsx",
    "25_model/25_model_9_9_0.xlsx",
    "25_model/25_model_9_9_2.xlsx",
    "25_model/25_model_9_9_5.xlsx"
]

# ===============================
# EXECUÇÃO
# ===============================
Z, z_preds, N = redes(file_names_25)

resultados = []

for lambda_reg in lambda_regs:
    for lr in lrs:
        for n_epocas in n_epocas_list:

            best_w = pesos(Z, z_preds, N, lambda_reg, lr, n_epocas)

            mse_25, r2_25, var_25, bias_25, covar_25 = avalia_ensemble(
                file_names_25, "25_model/25_model_12_6_0.xlsx", best_w
            )

            mse_1000, r2_1000, _, _, _ = avalia_ensemble(
                file_names_1000, "1000_model/1000_model_12_6_0.xlsx", best_w
            )

            resultados.append({
                "best_pesos": best_w,
                "lambda_reg": lambda_reg,
                "lr": lr,
                "n_epocas": n_epocas,
                "mse_25": mse_25,
                "mse_1000": mse_1000,
                "r2_25": r2_25,
                "r2_1000": r2_1000,
                "var_25": var_25,
                "bias_25": bias_25,
                "covar_25": covar_25
            })

lambda_regs = [1e-6, 5e-6, 1e-5, 1e-3]
lrs = [1e-5, 1e-2 , 1e-3]
n_epocas_list = [500000,5000000, 1000000]



tabela_resultados = pd.DataFrame(resultados)
pd.set_option('display.float_format', '{:.3e}'.format)
tabela_resultados["best_pesos"] = tabela_resultados["best_pesos"].apply(
    lambda w: "[" + ", ".join(f"{x:.6e}" for x in w) + "]"
)


tabela_resultados.to_excel(
    "tabela_resultados_completa.xlsx",
    index=False
)


In [7]:

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score


def redes(file_names):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list])

    dfZ = pd.read_excel("25_model/25_model_12_6_0.xlsx")
    Z = dfZ['Z'].values.reshape(-1, 1)

    N = Z.shape[0]
    return Z, z_preds, N

def pesos(Z, z_preds, N, lambda_reg, lr, n_epocas, patience=500, min_delta=1e-2):

    best_erro = np.inf
    best_w = None
    patience_counter = 0

    M = z_preds.shape[1]
    a = np.random.uniform(0, 1, M)  # parâmetros livres

    mu = 1e-3  # damping inicial

    for epoch in range(1, n_epocas + 1):

        # Softmax para impor restrições
        w = np.exp(a) / np.sum(np.exp(a))
        yhat = z_preds @ w

        residuo = Z.flatten() - yhat
        mse = np.mean(residuo ** 2)
        mse_pond = mse + lambda_reg * np.sum(a ** 2)

        # Jacobiano em relação a a (usando regra da cadeia)
        Jw = z_preds.T  # (M x N)
        s = Jw @ residuo  # (M,)

        # Gradiente em relação a a
        grad_a = w * (s - np.dot(s, w)) - 2 * lambda_reg * a

        # Hessiana aproximada (Gauss–Newton)
        H = np.outer(w, w) * (Jw @ Jw.T).mean() + 2 * lambda_reg * np.eye(M)

        # Atualização LM
        try:
            delta = np.linalg.solve(H + mu * np.eye(M), grad_a)
        except np.linalg.LinAlgError:
            break

        a_new = a + delta

        # Avalia novo erro
        w_new = np.exp(a_new) / np.sum(np.exp(a_new))
        yhat_new = z_preds @ w_new
        residuo_new = Z.flatten() - yhat_new
        mse_new = np.mean(residuo_new ** 2)
        mse_pond_new = mse_new + lambda_reg * np.sum(a_new ** 2)

        if mse_pond_new < mse_pond:
            a = a_new
            mu *= 0.7
        else:
            mu *= 2.0

        if mse_pond_new < best_erro - min_delta:
            best_erro = mse_pond_new
            best_w = w_new.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return best_w

def avalia_ensemble(file_names, Z_path, best_w):
    df_list = [pd.read_excel(f) for f in file_names]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list])

    Z = pd.read_excel(Z_path)['Z'].values.reshape(-1, 1)

    yhat = z_preds @ best_w
    mse = np.mean((Z.flatten() - yhat) ** 2)
    r2 = r2_score(Z, yhat)

    M = z_preds.shape[1]

    var = np.mean([(z_preds[:, i] - yhat) ** 2 for i in range(M)])
    bias = np.mean(yhat - Z.flatten())

    cov_sum = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                cov_sum += np.mean(
                    (z_preds[:, i] - z_preds[:, i].mean()) *
                    (z_preds[:, j] - z_preds[:, j].mean())
                )

    covar = cov_sum / (M * (M - 1))

    return mse, r2, var, bias, covar


file_names_1000 = [
    "1000_model/1000_model_12_6_0.xlsx",
    "1000_model/1000_model_13_9_6.xlsx",
    "1000_model/1000_model_20_5_0.xlsx",
    "1000_model/1000_model_22_8_7.xlsx",
    "1000_model/1000_model_24_6_2.xlsx",
    "1000_model/1000_model_2_8_0.xlsx",
    "1000_model/1000_model_30_7_8.xlsx",
    "1000_model/1000_model_9_9_0.xlsx",
    "1000_model/1000_model_9_9_2.xlsx",
    "1000_model/1000_model_9_9_5.xlsx"
]

file_names_25 = [
    "25_model/25_model_12_6_0.xlsx",
    "25_model/25_model_13_9_6.xlsx",
    "25_model/25_model_20_5_0.xlsx",
    "25_model/25_model_22_8_7.xlsx",
    "25_model/25_model_24_6_2.xlsx",
    "25_model/25_model_2_8_0.xlsx",
    "25_model/25_model_30_7_8.xlsx",
    "25_model/25_model_9_9_0.xlsx",
    "25_model/25_model_9_9_2.xlsx",
    "25_model/25_model_9_9_5.xlsx"
]

# ===============================
# EXECUÇÃO
# ===============================
Z, z_preds, N = redes(file_names_25)

resultados = []

for lambda_reg in lambda_regs:
    for lr in lrs:
        for n_epocas in n_epocas_list:

            best_w = pesos(Z, z_preds, N, lambda_reg, lr, n_epocas)

            mse_25, r2_25, var_25, bias_25, covar_25 = avalia_ensemble(
                file_names_25, "25_model/25_model_12_6_0.xlsx", best_w
            )

            mse_1000, r2_1000, _, _, _ = avalia_ensemble(
                file_names_1000, "1000_model/1000_model_12_6_0.xlsx", best_w
            )

            resultados.append({
                "best_pesos": best_w,
                "lambda_reg": lambda_reg,
                "lr": lr,
                "n_epocas": n_epocas,
                "mse_25": mse_25,
                "mse_1000": mse_1000,
                "r2_25": r2_25,
                "r2_1000": r2_1000,
                "var_25": var_25,
                "bias_25": bias_25,
                "covar_25": covar_25
            })

lambda_regs = [1e-1, 1e-2, 1e-3, 1e-4,1e-5, 1e-6, 1e-7, 1e-8]
lrs = [1e-1, 1e-2, 1e-3, 1e-4,1e-5, 1e-6, 1e-7, 1e-8]
n_epocas_list = [500000,5000000, 1000000, 500, 5000, 2000]



tabela_resultados = pd.DataFrame(resultados)
pd.set_option('display.float_format', '{:.3e}'.format)
tabela_resultados["best_pesos"] = tabela_resultados["best_pesos"].apply(
    lambda w: "[" + ", ".join(f"{x:.6e}" for x in w) + "]"
)


tabela_resultados.to_excel(
    "tabela_resultados_completa_lm.xlsx",
    index=False
)
